# 004. Polling an API without getting banned: OpenAQ /latest into Eventstream

Code from post 004 of the 365 days of Microsoft Fabric series, in the order the post runs it. The post explains each cell. The post publishes on t-sql.dk on Thursday 12 November 2026.

*Status: test plan with runnable code. The rate-limit headers and the shape of the `/latest` response have been confirmed against the live API. The worker below has not been run against Eventstream yet. The results of the first manual event and the first poll go into this post before it is published.*

In [ ]:
%pip install azure-eventhub

In [ ]:
# Parameters
VAULT_URL = "https://<your-vault-name>.vault.azure.net/"
LOCATION_IDS = [151, 2993, 10516, 4274, 10550, 625, 1948, 7570, 7998, 8118, 8039, 1214487, 5515, 784182]
WORKER_FOLDER = "/lakehouse/default/Files/worker"
METADATA_MAX_AGE_HOURS = 24
DRY_RUN = True  # True: print the events instead of sending them

In [ ]:
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path
import requests

API = "https://api.openaq.org/v3"
session = requests.Session()
session.headers["X-API-Key"] = notebookutils.credentials.getSecret(VAULT_URL, "openaq-api-key")
request_log = []

def openaq_get(path, attempts=3):
    for attempt in range(1, attempts + 1):
        started = time.perf_counter()
        try:
            response = session.get(f"{API}{path}", timeout=20)
        except requests.RequestException as error:
            request_log.append({"path": path, "status": type(error).__name__})
            time.sleep(2 ** attempt)
            continue
        headers = response.headers
        request_log.append({
            "path": path,
            "status": response.status_code,
            "ms": round((time.perf_counter() - started) * 1000),
            "remaining": headers.get("x-ratelimit-remaining"),
            "reset_s": headers.get("x-ratelimit-reset"),
        })
        if response.status_code == 200:
            if int(headers.get("x-ratelimit-remaining", "1")) <= 1:
                time.sleep(int(headers.get("x-ratelimit-reset", "60")) + 1)
            return response.json()["results"]
        if response.status_code == 429:
            time.sleep(int(headers.get("x-ratelimit-reset", "60")) + 1)
            continue
        if response.status_code in (401, 403):
            raise RuntimeError(f"OpenAQ refused the key ({response.status_code}). Fix it, do not retry.")
        if response.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        response.raise_for_status()
    raise RuntimeError(f"Gave up on {path} after {attempts} attempts")

def pause():
    time.sleep(random.uniform(0.5, 1.5))

In [ ]:
def load_sensor_metadata():
    cache = Path(WORKER_FOLDER) / "sensor_metadata.json"
    if cache.exists():
        cached = json.loads(cache.read_text())
        fresh = (time.time() - cached["saved_at"]) / 3600 < METADATA_MAX_AGE_HOURS
        if fresh and cached["location_ids"] == LOCATION_IDS:
            return {int(sensor_id): meta for sensor_id, meta in cached["sensors"].items()}
    sensors = {}
    for location_id in LOCATION_IDS:
        location = openaq_get(f"/locations/{location_id}")[0]
        for sensor in location["sensors"]:
            sensors[sensor["id"]] = {
                "provider": location["provider"]["name"],
                "parameter": sensor["parameter"]["name"],
                "unit": sensor["parameter"]["units"],
            }
        pause()
    cache.parent.mkdir(parents=True, exist_ok=True)
    cache.write_text(json.dumps({"saved_at": time.time(), "location_ids": LOCATION_IDS, "sensors": sensors}))
    return sensors

In [ ]:
def poll(sensors):
    polled_at = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
    raw, events = {}, []
    for location_id in LOCATION_IDS:
        rows = openaq_get(f"/locations/{location_id}/latest")
        raw[location_id] = rows
        for row in rows:
            meta = sensors.get(row["sensorsId"], {})
            events.append({
                "locationId": row["locationsId"],
                "sensorId": row["sensorsId"],
                "parameter": meta.get("parameter"),
                "unit": meta.get("unit"),
                "value": row["value"],
                "measurementTimeUtc": row["datetime"]["utc"],
                "measurementTimeLocal": row["datetime"]["local"],
                "polledAtUtc": polled_at,
                "provider": meta.get("provider"),
            })
        pause()
    raw_folder = Path(WORKER_FOLDER) / "raw"
    raw_folder.mkdir(parents=True, exist_ok=True)
    (raw_folder / f"{polled_at.replace(':', '')}.json").write_text(json.dumps(raw))
    return events

In [ ]:
from azure.eventhub import EventData, EventHubProducerClient

def send(events):
    connection = notebookutils.credentials.getSecret(VAULT_URL, "eventstream-connection")
    producer = EventHubProducerClient.from_connection_string(connection)
    with producer:
        batch = producer.create_batch()
        for event in events:
            data = EventData(json.dumps(event))
            try:
                batch.add(data)
            except ValueError:  # the batch is full
                producer.send_batch(batch)
                batch = producer.create_batch()
                batch.add(data)
        producer.send_batch(batch)

In [ ]:
send([{
    "locationId": 0, "sensorId": 0, "parameter": "pm25", "unit": "µg/m³", "value": 1.23,
    "measurementTimeUtc": "2026-11-12T09:00:00Z", "measurementTimeLocal": "2026-11-12T10:00:00+01:00",
    "polledAtUtc": "2026-11-12T09:10:00Z", "provider": "manual test",
}])

In [ ]:
sensors = load_sensor_metadata()
events = poll(sensors)
print(f"{len(events)} events from {len(LOCATION_IDS)} locations, {len(request_log)} HTTP requests")
for entry in request_log:
    print(entry)

if DRY_RUN:
    print(json.dumps(events[:3], indent=2, ensure_ascii=False))
else:
    send(events)